In [17]:
import sys
import os

# Add project root to Python path
sys.path.append(os.path.abspath(".."))

In [18]:
import pandas as pd
from backend.app.data_loader import load_pantry, load_recipes

pantry = load_pantry()
recipes = load_recipes()

pantry, recipes

(  ingredient  quantity unit
 0      onion       500    g
 1       milk      1000   ml
 2     paneer       200    g
 3        oil       250   ml,
                  recipe       meal difficulty       type
 0          Veg Omelette  breakfast       easy    healthy
 1  Paneer Butter Masala      lunch     medium  indulgent
 2      Chicken Stir Fry     dinner       easy    healthy)

In [19]:
def simple_coverage_score(pantry, recipe_ingredients):
    # for now, fake example
    return 0.75

print("Coverage score:", simple_coverage_score(pantry, None))


Coverage score: 0.75


In [20]:
from backend.app.db import SessionLocal
from backend.app.models import Recipe

db = SessionLocal()

recipes = db.query(Recipe).all()

print(f"Total recipes: {len(recipes)}")

for r in recipes:
    print(r.id, r.title)
    
db.close()


Total recipes: 3
1 Veg Omelette
2 Paneer Butter Masala
3 Chicken Stir Fry


In [21]:
from backend.app.db import SessionLocal
from backend.app.models import Recipe

db = SessionLocal()

recipe = db.query(Recipe).first()

print("Recipe:", recipe.title)
print("Ingredients:")

for ri in recipe.ingredients:
    print(
        ri.ingredient.name,
        ri.qty,
        ri.unit,
        "Optional:", ri.optional
    )

db.close()


Recipe: Veg Omelette
Ingredients:
eggs 3.0 pcs Optional: False
onion 50.0 g Optional: False
oil 10.0 ml Optional: False


In [22]:
from sqlalchemy.orm import joinedload

from backend.app.data_loader import load_pantry
from backend.app.db import SessionLocal
from backend.app.models import Recipe, RecipeIngredient
from backend.app.services.serializers import recipe_to_dict
from backend.app.services.matching import compute_coverage

pantry = load_pantry()

db = SessionLocal()
recipes = (
    db.query(Recipe)
    .options(
        joinedload(Recipe.ingredients).joinedload(RecipeIngredient.ingredient)
    )
    .all()
)
db.close()

for r in recipes:
    rd = recipe_to_dict(r)
    res = compute_coverage(pantry, rd)
    print(rd["title"], "=>", res.coverage_pct, "%", "Missing required:", [m["name"] for m in res.missing_required])



Veg Omelette => 66.67 % Missing required: ['eggs']
Paneer Butter Masala => 100.0 % Missing required: []
Chicken Stir Fry => 100.0 % Missing required: []


In [23]:
from backend.app.data_loader import load_pantry
pantry = load_pantry()
pantry


,ingredient,quantity,unit
0,onion,500,g
1,milk,1000,ml
2,paneer,200,g
3,oil,250,ml


In [24]:
print(pantry.columns)
print(pantry.head(20))


Index(['ingredient', 'quantity', 'unit'], dtype='str')
  ingredient  quantity unit
0      onion       500    g
1       milk      1000   ml
2     paneer       200    g
3        oil       250   ml


In [25]:
pantry3 = pantry.copy()
mask = pantry3["ingredient"].str.lower() == "eggs"
pantry3.loc[mask, "quantity"] = 0

res = compute_coverage(pantry3, recipe_to_dict(recipes[0]))
res.coverage_pct, res.missing_required


(66.67,
 [{'name': 'eggs',
   'needed_qty': 3.0,
   'unit': 'pcs',
   'available_qty': 0.0,
   'available_unit': None}])

In [32]:
from sqlalchemy.orm import joinedload

from backend.app.db import SessionLocal
from backend.app.models import Recipe, RecipeIngredient
from backend.app.services.recommend import recommend_recipes
from backend.app.data_loader import load_pantry

pantry = load_pantry()

db = SessionLocal()
recipes = (
    db.query(Recipe)
    .options(
        joinedload(Recipe.ingredients).joinedload(RecipeIngredient.ingredient)
    )
    .all()
)
db.close()

recommend_recipes(pantry, recipes, top_k=3)

[{'id': '1',
  'title': 'Veg Omelette',
  'coverage_pct': 100.0,
  'missing_required': [],
  'missing_optional': [],
  'score': 100.0},
 {'id': '2',
  'title': 'Paneer Butter Masala',
  'coverage_pct': 100.0,
  'missing_required': [],
  'missing_optional': [],
  'score': 100.0},
 {'id': '3',
  'title': 'Chicken Stir Fry',
  'coverage_pct': 100.0,
  'missing_required': [],
  'missing_optional': [],
  'score': 100.0}]

In [33]:
from backend.app.services.matching import compute_coverage
from backend.app.services.serializers import recipe_to_dict

# Use the same recipes list you already queried from DB
omelette = recipe_to_dict(recipes[0])  # Veg Omelette (assuming it's first)

# 1) Remove eggs from pantry
pantry_no_eggs = pantry[pantry["ingredient"].str.lower() != "eggs"].copy()
res1 = compute_coverage(pantry_no_eggs, omelette)
print("No eggs coverage:", res1.coverage_pct, "missing:", [m["name"] for m in res1.missing_required])

# 2) Set oil quantity to 0
pantry_oil0 = pantry.copy()
mask = pantry_oil0["ingredient"].str.lower() == "oil"
pantry_oil0.loc[mask, "quantity"] = 0
res2 = compute_coverage(pantry_oil0, omelette)
print("Oil=0 coverage:", res2.coverage_pct, "missing:", [m["name"] for m in res2.missing_required])


No eggs coverage: 66.67 missing: ['eggs']
Oil=0 coverage: 66.67 missing: ['oil']


In [39]:
import importlib
from backend.app.services import recommend
importlib.reload(recommend)
from backend.app.services.recommend import recommend_recipes
recommend_recipes(pantry, recipes, exclude_ingredients=["eggs"])

[{'id': '2',
  'title': 'Paneer Butter Masala',
  'coverage_pct': 100.0,
  'score': 100.0,
  'missing_required': []},
 {'id': '3',
  'title': 'Chicken Stir Fry',
  'coverage_pct': 100.0,
  'score': 100.0,
  'missing_required': []}]

In [37]:
recommend_recipes(pantry, recipes, meal_type="breakfast")


[{'id': '1',
  'title': 'Veg Omelette',
  'coverage_pct': 100.0,
  'score': 100.0,
  'missing_required': []}]

In [40]:
recommend_recipes(pantry, recipes, preferred_cuisine=["indian"])

[{'id': '1',
  'title': 'Veg Omelette',
  'coverage_pct': 100.0,
  'score': 100.0,
  'missing_required': []},
 {'id': '2',
  'title': 'Paneer Butter Masala',
  'coverage_pct': 100.0,
  'score': 100.0,
  'missing_required': []}]